# video_studio — установка на Google Colab (T4)

Обоснование каждого шага — в `COLAB_MIGRATION_PLAN.md` (разделы 2 и 4) в корне репозитория.

Порядок ячеек важен: сначала Drive и `VIDEO_STUDIO_HOME` (чтобы модели/проекты не терялись при пересоздании рантайма), потом клонирование репозитория, потом установка зависимостей.

**Runtime → Change runtime type → T4 GPU** должен быть выбран ДО запуска ячеек.

In [ ]:
# 1. Проверяем, что GPU реально выдан этому рантайму
!nvidia-smi

In [ ]:
# 2. Google Drive — единая точка хранения проектов/моделей/логов между сессиями
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["VIDEO_STUDIO_HOME"] = "/content/drive/MyDrive/video_studio_home"
os.environ["HF_HOME"] = os.environ["VIDEO_STUDIO_HOME"] + "/models_cache/hf"
os.environ["TORCH_HOME"] = os.environ["VIDEO_STUDIO_HOME"] + "/models_cache/torch"
os.makedirs(os.environ["VIDEO_STUDIO_HOME"], exist_ok=True)
print("VIDEO_STUDIO_HOME =", os.environ["VIDEO_STUDIO_HOME"])

In [ ]:
# 3. Клонирование репозитория
REPO_URL = "https://github.com/AeVideo/video_studio.git"

!git clone $REPO_URL /content/video_studio
%cd /content/video_studio

# При повторном запуске в новой сессии — если /content/video_studio уже есть,
# просто обновите вместо повторного клонирования:
# %cd /content/video_studio && !git pull

In [ ]:
# 4. torch/torchaudio/torchvision — ОТДЕЛЬНОЙ командой с CUDA-индексом.
# Версия cu-индекса намеренно не фиксируется в requirements-gpu.txt заранее
# (см. COLAB_MIGRATION_PLAN.md, раздел 7) — если у выданного вам инстанса
# другая версия CUDA, замените cu121 на подходящую (see pytorch.org/get-started).
!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121

import torch
print("torch:", torch.__version__, "| CUDA доступна:", torch.cuda.is_available(), "| устройство:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

In [ ]:
# 5. Остальные зависимости + сам пакет video_studio (editable)
!pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
# 6. Секреты — через Colab Secrets (значок ключа слева), НЕ через .env на Drive
# (см. COLAB_MIGRATION_PLAN.md, раздел 4.1 — почему не .env)
from google.colab import userdata

for key in ("DEEPSEEK_API_KEY", "PEXELS_API_KEY", "PIXABAY_API_KEY", "ELEVENLABS_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"! {key} не задан в Colab Secrets — добавьте через значок ключа слева, если понадобится")

In [ ]:
# 7. Запуск Gradio-интерфейса (Phase 2, web/gradio_app.py)
# share=True — публичная ссылка, доступная без деплоя, живёт пока жив рантайм ноутбука
!python -m web.gradio_app